# Module 4.2: Volatility Regime Detection with VIX

## Learning Objectives
By completing this notebook, you will:
1. Model VIX (volatility index) with regime-switching HMMs
2. Identify low, medium, and high volatility regimes
3. Analyze volatility persistence and mean reversion
4. Build volatility forecasting models
5. Link volatility regimes to market returns

## Prerequisites
- Market regime detection (Module 4.1)
- Volatility modeling basics

## Estimated Time: 50 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Market regime detection (Module 4.1)",
    "Volatility modeling basics"
])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Generate Realistic VIX Data

VIX characteristics:
- Mean-reverting
- Skewed distribution
- Regime-dependent behavior

In [ ]:
def generate_vix_data(T=2520, random_seed=42):
    np.random.seed(random_seed)
    regimes = {
        0: {'name': 'Low-Vol', 'mean': 12, 'std': 2},
        1: {'name': 'Medium-Vol', 'mean': 20, 'std': 4},
        2: {'name': 'High-Vol', 'mean': 35, 'std': 8}
    }
    A = np.array([[0.98, 0.015, 0.005],
                  [0.05, 0.92, 0.03],
                  [0.02, 0.08, 0.90]])
    pi = np.array([0.70, 0.25, 0.05])
    states = np.zeros(T, dtype=int)
    states[0] = np.random.choice(3, p=pi)
    for t in range(1, T):
        states[t] = np.random.choice(3, p=A[states[t-1]])
    vix = np.zeros(T)
    for t in range(T):
        r = regimes[states[t]]
        vix[t] = np.abs(np.random.normal(r['mean'], r['std']))
    dates = pd.date_range('2015-01-01', periods=T, freq='B')
    df = pd.DataFrame({'Date': dates, 'VIX': vix, 'True_State': states})
    return df

vix_df = generate_vix_data()
print(f"VIX data: {len(vix_df)} days")
print(vix_df['VIX'].describe())

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(vix_df['Date'], vix_df['VIX'], linewidth=1)
ax.set_title('VIX Index', fontsize=14)
ax.set_ylabel('VIX Level', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Fit HMM to VIX

Model VIX levels with 3-state Gaussian HMM.

In [ ]:
class GaussianHMM:
    def __init__(self, n_states):
        self.K = n_states
        self.pi = np.ones(self.K) / self.K
        self.A = np.random.rand(self.K, self.K)
        self.A = self.A / self.A.sum(axis=1, keepdims=True)
        self.means = None
        self.stds = None
    
    def fit(self, obs, max_iter=50):
        T = len(obs)
        quantiles = np.linspace(0, 100, self.K+1)
        self.means = np.array([np.percentile(obs, (quantiles[i]+quantiles[i+1])/2) for i in range(self.K)])
        self.stds = np.ones(self.K) * obs.std() * 0.5
        for _ in range(max_iter):
            alpha = np.zeros((T, self.K))
            scaling = np.zeros(T)
            alpha[0] = self.pi * stats.norm.pdf(obs[0], self.means, self.stds)
            scaling[0] = alpha[0].sum()
            alpha[0] /= scaling[0]
            for t in range(1, T):
                for j in range(self.K):
                    alpha[t,j] = (alpha[t-1] @ self.A[:,j]) * stats.norm.pdf(obs[t], self.means[j], self.stds[j])
                scaling[t] = alpha[t].sum()
                alpha[t] /= scaling[t]
            beta = np.zeros((T, self.K))
            beta[-1] = 1/scaling[-1]
            for t in range(T-2, -1, -1):
                for i in range(self.K):
                    beta[t,i] = sum(self.A[i,j] * stats.norm.pdf(obs[t+1], self.means[j], self.stds[j]) * beta[t+1,j] for j in range(self.K))
                beta[t] /= scaling[t]
            gamma = alpha * beta
            gamma /= gamma.sum(axis=1, keepdims=True)
            self.pi = gamma[0]
            for i in range(self.K):
                for j in range(self.K):
                    self.A[i,j] = sum([alpha[t,i] * self.A[i,j] * stats.norm.pdf(obs[t+1], self.means[j], self.stds[j]) * beta[t+1,j] for t in range(T-1)]) / gamma[:-1,i].sum()
            self.A /= self.A.sum(axis=1, keepdims=True)
            for i in range(self.K):
                self.means[i] = (gamma[:,i] * obs).sum() / gamma[:,i].sum()
                self.stds[i] = np.sqrt((gamma[:,i] * (obs - self.means[i])**2).sum() / gamma[:,i].sum())
    
    def predict(self, obs):
        T = len(obs)
        delta = np.zeros((T, self.K))
        psi = np.zeros((T, self.K), dtype=int)
        delta[0] = np.log(self.pi + 1e-10) + np.log(stats.norm.pdf(obs[0], self.means, self.stds) + 1e-10)
        for t in range(1, T):
            for j in range(self.K):
                probs = delta[t-1] + np.log(self.A[:,j] + 1e-10)
                psi[t,j] = probs.argmax()
                delta[t,j] = probs[psi[t,j]] + np.log(stats.norm.pdf(obs[t], self.means[j], self.stds[j]) + 1e-10)
        states = np.zeros(T, dtype=int)
        states[-1] = delta[-1].argmax()
        for t in range(T-2, -1, -1):
            states[t] = psi[t+1, states[t+1]]
        return states

hmm_vix = GaussianHMM(3)
hmm_vix.fit(vix_df['VIX'].values)
decoded = hmm_vix.predict(vix_df['VIX'].values)
print(f"Means: {hmm_vix.means}")
print(f"Stds: {hmm_vix.stds}")

## Exercise 1: Volatility Regime Analysis

In [ ]:
# YOUR CODE HERE
def analyze_vix_regimes(vix, states):
    # Compute statistics per regime
    return None

regime_stats = analyze_vix_regimes(vix_df['VIX'].values, decoded)

In [ ]:
def test_exercise_1():
    assert regime_stats is not None
    print("✅ Exercise 1 passed!")
test_exercise_1()

## Exercise 2: Volatility Forecasting

In [ ]:
# YOUR CODE HERE
def forecast_volatility(hmm, current_state, horizon=5):
    # Multi-step ahead forecast
    return None

forecasts = forecast_volatility(hmm_vix, decoded[-1], 5)

In [ ]:
def test_exercise_2():
    assert forecasts is not None
    print("✅ Exercise 2 passed!")
test_exercise_2()

## Exercise 3: VIX-Return Relationship

In [ ]:
# YOUR CODE HERE
def analyze_vix_return_relationship(vix_regimes, market_returns):
    # Analyze correlation structure
    return None

relationship = analyze_vix_return_relationship(decoded, vix_df['VIX'].pct_change().values)

In [ ]:
def test_exercise_3():
    assert relationship is not None
    print("✅ Exercise 3 passed!")
test_exercise_3()

## Summary

### Key Takeaways
1. VIX exhibits clear regime structure
2. Volatility regimes are highly persistent
3. High-vol regimes coincide with market stress
4. Regime-based forecasting improves accuracy
5. VIX inversely correlated with market returns

---